# Modelo baseline — Riesgo de incumplimiento de SLA (F1-05)

**Objetivo:** entrenar el modelo más simple posible (regresión logística) sobre `train.parquet`/`test.parquet` (ya separados por ruta, estratificados por `route_score`, en `01_eda_inicial.ipynb`), para tener un piso de comparación antes de probar modelos más complejos (LightGBM, Random Forest — F1-06).

**Por qué un baseline primero:** si más adelante un modelo complejo apenas mejora estos resultados, es una señal de que la complejidad no está aportando mucho. Sin este piso, no hay con qué comparar.

In [7]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

PROCESSED = "../../data/processed"

train_df = pd.read_parquet(f"{PROCESSED}/train.parquet")
test_df = pd.read_parquet(f"{PROCESSED}/test.parquet")
print("Train:", train_df.shape, "| Test:", test_df.shape)

Train: (719870, 27) | Test: (178545, 27)


## 1. Selección de features

**Se descartan explícitamente:**

- `route_id`, `stop_id`: identificadores únicos por fila (o casi). Si se codificaran como categoría, el modelo memorizaría "esta ruta puntual dio tal resultado" en vez de aprender un patrón real — información inútil para una ruta nueva, que siempre tiene un ID que el modelo nunca vio. Mismo tipo de problema que el leakage ya visto, aplicado a nivel feature.
- `date`, `departure_time_utc`: podrían tener señal real (estacionalidad, día de semana), pero como texto crudo no la aportan — quedan pendientes de un feature engineering que no se hizo todavía.
- `lat`, `lng`: coordenadas crudas sobre las que un modelo **lineal** solo puede aprender relaciones lineales ("más al norte, más riesgo"), que no reflejan cómo funciona el riesgo real (específico de cada zona, no un gradiente). `zona_riesgo_low` ya resuelve esta señal geográfica correctamente.
- `zone_id`: reemplazado por `zona_riesgo_low`, que expresa la misma información como tasa de riesgo en vez de como texto categórico con miles de valores.
- `window_start_utc`, `window_end_utc`: ya se extrajo lo útil de estas columnas (`window_duration_min`, `franja_horaria`) en el EDA.
- `stop_type`: un solo valor posible en toda la tabla (`Dropoff`, ya filtrado en el EDA) — columna sin variación, cero poder predictivo.
- `seq_order`: posición de la parada dentro de la secuencia de la ruta, sin relación directa con el riesgo de SLA.

**Se mantiene `station_code`** a pesar de ser texto: a diferencia de `route_id`, solo tiene 17 valores distintos que se repiten en cientos de rutas cada uno — un patrón real y generalizable ("las rutas de esta estación tienden a..."), no un identificador único.

In [8]:
feature_cols_num = [
    "n_packages", "total_volume_cm3", "total_planned_service_seconds",
    "window_duration_min", "volumen_promedio_paquete_cm3",
    "paradas_por_ruta", "paquetes_por_ruta", "distancia_a_siguiente_km",
    "zona_riesgo_low",
]
feature_cols_bool = ["has_time_window", "any_rejected", "any_attempted"]
feature_cols_cat = ["station_code", "franja_horaria"]

print(f"Total features: {len(feature_cols_num) + len(feature_cols_bool) + len(feature_cols_cat)}")

Total features: 14


## 2. Nulos: rellenar con significado de negocio, no inventar ni descartar

Tres columnas tienen `NaN`, cada una por un motivo distinto — y la regresión logística no acepta `NaN` en la entrada:

- **`window_duration_min`** (93.4% nulo): la parada no tenía ventana horaria. Se rellena con `0` — no se confunde con una ventana real porque la más corta dura 210 minutos, y además `has_time_window` ya marca esta situación por separado.
- **`franja_horaria`** (mismo 93.4%): se agrega una categoría nueva explícita, `"sin_ventana"`, en vez de imputar una franja horaria inventada (fabricar ese dato sería repetir el error que se descartó en el EDA al definir `window_duration_min`).
- **`distancia_a_siguiente_km`** (6,112 filas — la última parada de cada ruta no tiene una parada siguiente dentro de la ruta): se rellena con la **mediana calculada solo con train**, aplicada también a test — mismo principio de no-leakage que se usó para `zona_riesgo_low`.

**Importante:** la mediana de distancia se calcula una sola vez sobre train y ese mismo valor se aplica a test — si se recalculara en test, sería la misma fuga de datos ya evitada en la sección de zona.

In [9]:
def prep(df, median_dist=None):
    df = df.copy()
    df["window_duration_min"] = df["window_duration_min"].fillna(0)
    df["franja_horaria"] = df["franja_horaria"].cat.add_categories("sin_ventana").fillna("sin_ventana")
    if median_dist is None:
        median_dist = df["distancia_a_siguiente_km"].median()
    df["distancia_a_siguiente_km"] = df["distancia_a_siguiente_km"].fillna(median_dist)
    for c in feature_cols_bool:
        df[c] = df[c].astype(int)
    return df, median_dist


train_df, median_dist = prep(train_df)
test_df, _ = prep(test_df, median_dist=median_dist)

print("Mediana de distancia (train, usada como fallback):", round(median_dist, 4), "km")
all_feats = feature_cols_num + feature_cols_bool + feature_cols_cat
print("Nulos restantes en train (debe ser 0):", train_df[all_feats].isna().sum().sum())
print("Nulos restantes en test (debe ser 0):", test_df[all_feats].isna().sum().sum())

Mediana de distancia (train, usada como fallback): 0.1341 km
Nulos restantes en train (debe ser 0): 0
Nulos restantes en test (debe ser 0): 0


## 3. Codificación de categorías: one-hot, no números arbitrarios

`station_code` y `franja_horaria` son categorías (texto), no números — la regresión logística necesita todo en formato numérico. La tentación de junior es asignar números arbitrarios (madrugada=1, mañana=2...), pero eso le mete al modelo una relación matemática falsa: que "noche" (4) es el cuádruple de "madrugada" (1), y que la distancia entre categorías es siempre la misma. Son categorías sin ese orden proporcional.

La solución es **one-hot encoding**: una columna binaria (0/1) por cada categoría posible, en vez de un solo número. `pd.get_dummies` hace esto automáticamente.

**Detalle técnico importante:** `X_test` se reindexa con las columnas exactas de `X_train` (`reindex(columns=X_train.columns, fill_value=0)`). Esto evita un problema real: si alguna categoría de `station_code` apareciera en test pero no en train (o viceversa), `get_dummies` generaría un número distinto de columnas en cada tabla, y el modelo no podría aplicarse a test.

In [10]:
X_train = pd.get_dummies(train_df[all_feats], columns=feature_cols_cat)
X_test = pd.get_dummies(test_df[all_feats], columns=feature_cols_cat)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

y_train = train_df["route_score"]
y_test = test_df["route_score"]

print("X_train:", X_train.shape, "| X_test:", X_test.shape)

X_train: (719870, 34) | X_test: (178545, 34)


## 4. Entrenamiento

Dos decisiones más antes de entrenar:

- **Escalado (`StandardScaler`):** la regresión logística es sensible a la escala de las variables — `total_volume_cm3` llega a valores de cientos de miles, mientras que `has_time_window` es 0 o 1. Sin escalar, las variables con números grandes dominan el entrenamiento no porque sean más importantes, sino por su magnitud.
- **`class_weight="balanced"`:** ya sabemos que `Low` es el 1.7% de los datos. Sin esto, el modelo puede lograr una accuracy alta simplemente casi no prediciendo nunca `Low` (total, acertaría casi siempre con `Medium`/`High` y accuracy se vería "bien"). Este parámetro le da más peso al error de la clase minoritaria durante el entrenamiento, para que no la ignore.

In [11]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = LogisticRegression(max_iter=300, class_weight="balanced")
model.fit(X_train_s, y_train)
print("Modelo entrenado.")

Modelo entrenado.


## 5. Evaluación

**Por qué no alcanza con accuracy:** con `Low` en 1.7% de los datos, un modelo que jamás prediga `Low` tendría ~85% de accuracy y sería inútil para lo único que de verdad importa (detectar rutas de riesgo bajo). Por eso se mira `classification_report` (precision/recall/f1 por clase) y la matriz de confusión, no un solo número. Se suma también el ROC-AUC (promedio one-vs-rest de las 3 clases) para poder comparar contra Random Forest (F1-06) con la misma métrica.

In [12]:
y_pred = model.predict(X_test_s)
y_proba = model.predict_proba(X_test_s)

print(classification_report(y_test, y_pred))
print("Matriz de confusión (filas=real, columnas=predicho), orden Low/Medium/High:")
print(confusion_matrix(y_test, y_pred, labels=["Low", "Medium", "High"]))

auc = roc_auc_score(y_test, y_proba, multi_class="ovr", labels=model.classes_)
print("ROC-AUC (ovr, macro):", round(auc, 4))

              precision    recall  f1-score   support

        High       0.55      0.50      0.52     74901
         Low       0.02      0.18      0.04      3047
      Medium       0.68      0.57      0.62    100597

    accuracy                           0.53    178545
   macro avg       0.42      0.41      0.39    178545
weighted avg       0.62      0.53      0.57    178545

Matriz de confusión (filas=real, columnas=predicho), orden Low/Medium/High:
[[  549  1326  1172]
 [14925 56896 28776]
 [12477 25122 37302]]
ROC-AUC (ovr, macro): 0.6102


## 6. Conclusión de negocio

**En una línea:** el modelo baseline distingue razonablemente bien rutas `Medium` y `High` (F1 ~0.6 y ~0.5), pero todavía **no es confiable para detectar rutas `Low`** — de cada 100 rutas realmente `Low`, identifica correctamente unas 18, aun después de compensar el desbalance de clases con `class_weight`. Para un negocio de logística, esto significa que hoy el modelo dejaría pasar la mayoría de las rutas de mayor riesgo sin marcarlas — sirve como piso de comparación, no como herramienta operativa todavía.

**Nota sobre `zona_riesgo_low`:** esta versión ya usa la variable de zona con suavizado bayesiano (corregido tras una pasada de QA — ver README y `01_eda_inicial.ipynb`, sección 14). El número de `Low` cambió apenas respecto a la versión sin suavizar (16% → 18% de recall) — la corrección era necesaria por rigor metodológico (evitar que zonas con 1 sola ruta de historia parezcan 100% riesgosas), pero no resultó ser la palanca que mejora el modelo de fondo.

**Pendiente (F1-06):** probar un modelo no lineal (LightGBM o Random Forest) que pueda capturar interacciones entre variables que la regresión logística, al ser lineal, no puede — y evaluar si mejora específicamente el recall de la clase `Low`, no solo la accuracy general.